
# Task 2 — Preprocessing

This notebook prepares the 10% English–Italian Europarl sample for the neural machine translation models used in the next tasks.

The preprocessing is intentionally conservative. For machine translation, the target text must remain as close as possible to natural language: punctuation, numbers, stop words, and inflected word forms may all be useful for translation. Therefore, we apply only the steps that clean obvious noise and reduce unnecessary variation:

1. remove aligned pairs where either side is empty;
2. remove aligned pairs where either side starts with an XML tag (`<...`);
3. lowercase both languages;
4. normalize repeated whitespace.

The output files are saved under `data/processed/` and should be used as the input for Task 3.



## 1. Imports and project paths

The notebook is written so that it works both when launched from the project root and from the `notebooks/` folder.
It also reads either the uncompressed files (`sample.en`, `sample.it`) or the gzipped files (`sample.en.gz`, `sample.it.gz`).


In [1]:
from pathlib import Path
import gzip
import json
import re
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the repository root by looking for the data/sample folder."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "sample").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Please run this notebook inside the project folder."
    )


ROOT = find_project_root()
SAMPLE_DIR = ROOT / "data" / "sample"
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results" / "task2_outputs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EN_PATH = SAMPLE_DIR / "sample.en"
IT_PATH = SAMPLE_DIR / "sample.it"
EN_GZ_PATH = SAMPLE_DIR / "sample.en.gz"
IT_GZ_PATH = SAMPLE_DIR / "sample.it.gz"

print(f"Project root: {ROOT}")
print(f"Sample folder: {SAMPLE_DIR}")
print(f"Processed output folder: {PROCESSED_DIR}")

Project root: /home/vasilis/Desktop/English-Italian Machine Translation
Sample folder: /home/vasilis/Desktop/English-Italian Machine Translation/data/sample
Processed output folder: /home/vasilis/Desktop/English-Italian Machine Translation/data/processed



## 2. Read the aligned sample

The two files must contain the same number of lines, because each English line corresponds to the Italian line with the same index.


In [2]:
def pick_existing_path(uncompressed_path: Path, gz_path: Path) -> Path:
    """Use the uncompressed file if available, otherwise fall back to the gzipped file."""
    if uncompressed_path.exists():
        return uncompressed_path
    if gz_path.exists():
        return gz_path
    raise FileNotFoundError(
        f"Missing both {uncompressed_path} and {gz_path}. "
        "Make sure the 10% sample is available in data/sample/."
    )


def read_lines(path: Path) -> list[str]:
    """Read a plain text or .gz text file as UTF-8."""
    if path.suffix == ".gz":
        with gzip.open(path, "rt", encoding="utf-8") as f:
            return [line.rstrip("\n") for line in f]
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]


en_input_path = pick_existing_path(EN_PATH, EN_GZ_PATH)
it_input_path = pick_existing_path(IT_PATH, IT_GZ_PATH)

en_raw = read_lines(en_input_path)
it_raw = read_lines(it_input_path)

assert len(en_raw) == len(it_raw), "The English and Italian files are not aligned."

print(f"English input file: {en_input_path.relative_to(ROOT)}")
print(f"Italian input file: {it_input_path.relative_to(ROOT)}")
print(f"Aligned sentence pairs: {len(en_raw):,}")

English input file: data/sample/sample.en
Italian input file: data/sample/sample.it
Aligned sentence pairs: 190,912



## 3. Define preprocessing rules

The rules are deliberately simple and transparent:

- `strip_and_normalize_spaces`: removes leading/trailing spaces and collapses repeated spaces/tabs into one space;
- `starts_with_xml_tag`: detects lines that begin with `<` after removing leading spaces;
- `preprocess_text`: applies whitespace normalization and lowercasing.

We do **not** remove punctuation, numbers, stop words, or inflectional suffixes, because they can carry information that is useful for translation.


In [3]:
LOWERCASE = True
REMOVE_EMPTY_PAIRS = True
REMOVE_XML_TAG_PAIRS = True


def strip_and_normalize_spaces(text: str) -> str:
    """Strip leading/trailing spaces and collapse repeated whitespace."""
    return re.sub(r"\s+", " ", text).strip()


def starts_with_xml_tag(text: str) -> bool:
    """Return True if a line starts with an XML-like tag after leading spaces."""
    return text.lstrip().startswith("<")


def preprocess_text(text: str, lowercase: bool = True) -> str:
    """Apply the selected preprocessing steps to a single sentence."""
    text = strip_and_normalize_spaces(text)
    if lowercase:
        text = text.lower()
    return text


print("Preprocessing configuration:")
print(f"- Lowercase: {LOWERCASE}")
print(f"- Remove empty aligned pairs: {REMOVE_EMPTY_PAIRS}")
print(f"- Remove XML-tag aligned pairs: {REMOVE_XML_TAG_PAIRS}")

Preprocessing configuration:
- Lowercase: True
- Remove empty aligned pairs: True
- Remove XML-tag aligned pairs: True



## 4. Inspect removable aligned pairs

For a parallel corpus, removal must happen at pair level. If one side is empty or XML-like, both the English and Italian lines are removed to preserve alignment.


In [4]:
df = pd.DataFrame({
    "en_raw": en_raw,
    "it_raw": it_raw,
})

df["en_stripped"] = df["en_raw"].map(strip_and_normalize_spaces)
df["it_stripped"] = df["it_raw"].map(strip_and_normalize_spaces)

df["en_empty"] = df["en_stripped"].eq("")
df["it_empty"] = df["it_stripped"].eq("")
df["en_xml"] = df["en_stripped"].map(starts_with_xml_tag)
df["it_xml"] = df["it_stripped"].map(starts_with_xml_tag)

df["remove_empty_pair"] = df["en_empty"] | df["it_empty"]
df["remove_xml_pair"] = df["en_xml"] | df["it_xml"]

df["remove_pair"] = False
if REMOVE_EMPTY_PAIRS:
    df["remove_pair"] = df["remove_pair"] | df["remove_empty_pair"]
if REMOVE_XML_TAG_PAIRS:
    df["remove_pair"] = df["remove_pair"] | df["remove_xml_pair"]

summary = pd.DataFrame({
    "count": [
        len(df),
        int(df["en_empty"].sum()),
        int(df["it_empty"].sum()),
        int(df["remove_empty_pair"].sum()),
        int(df["en_xml"].sum()),
        int(df["it_xml"].sum()),
        int(df["remove_xml_pair"].sum()),
        int(df["remove_pair"].sum()),
        int((~df["remove_pair"]).sum()),
    ]
}, index=[
    "total_pairs_before_preprocessing",
    "empty_english_lines",
    "empty_italian_lines",
    "pairs_removed_because_empty",
    "xml_like_english_lines",
    "xml_like_italian_lines",
    "pairs_removed_because_xml_like",
    "total_pairs_removed",
    "total_pairs_after_preprocessing",
])

summary["percentage"] = 100 * summary["count"] / len(df)
summary.loc["total_pairs_before_preprocessing", "percentage"] = 100.0
summary.loc["total_pairs_after_preprocessing", "percentage"] = 100 * summary.loc["total_pairs_after_preprocessing", "count"] / len(df)
summary["percentage"] = summary["percentage"].round(4)

summary

,count,percentage
total_pairs_before_preprocessing,190912,100.0000
empty_english_lines,326,0.1708
empty_italian_lines,476,0.2493
pairs_removed_because_empty,802,0.4201
xml_like_english_lines,0,0.0000
xml_like_italian_lines,0,0.0000
pairs_removed_because_xml_like,0,0.0000
total_pairs_removed,802,0.4201
total_pairs_after_preprocessing,190110,99.5799



## 5. Apply preprocessing and keep alignment

After removing invalid pairs, the English and Italian lists must still have the same length.


In [5]:
clean_df = df.loc[~df["remove_pair"], ["en_raw", "it_raw"]].copy()

clean_df["en"] = clean_df["en_raw"].map(lambda text: preprocess_text(text, lowercase=LOWERCASE))
clean_df["it"] = clean_df["it_raw"].map(lambda text: preprocess_text(text, lowercase=LOWERCASE))

# Safety check after preprocessing: no empty lines should remain.
clean_df = clean_df[(clean_df["en"] != "") & (clean_df["it"] != "")].copy()
clean_df = clean_df.reset_index(drop=True)

assert clean_df["en"].notna().all()
assert clean_df["it"].notna().all()
assert len(clean_df["en"]) == len(clean_df["it"])

print(f"Pairs before preprocessing: {len(df):,}")
print(f"Pairs after preprocessing:  {len(clean_df):,}")
print(f"Pairs removed:              {len(df) - len(clean_df):,}")

clean_df[["en_raw", "en", "it_raw", "it"]].head(5)

Pairs before preprocessing: 190,912
Pairs after preprocessing:  190,110
Pairs removed:              802


,en_raw,en,it_raw,it
0,"Madam President, on a point of order.","madam president, on a point of order.","Signora Presidente, un richiamo al Regolamento.","signora presidente, un richiamo al regolamento."
1,"Madam President, coinciding with this year' s first part-session of the European Parliament, a date has been set, un...","madam president, coinciding with this year' s first part-session of the european parliament, a date has been set, un...","Signora Presidente, in coincidenza con la prima tornata dell'anno del Parlamento europeo, negli Stati Uniti in Texas...","signora presidente, in coincidenza con la prima tornata dell'anno del parlamento europeo, negli stati uniti in texas..."
2,"Madam President, I would firstly like to compliment you on the fact that you have kept your word and that, during th...","madam president, i would firstly like to compliment you on the fact that you have kept your word and that, during th...","Signora Presidente, mi permetta di farle innanzi tutto i miei complimenti per aver tenuto fede alla parola data. In ...","signora presidente, mi permetta di farle innanzi tutto i miei complimenti per aver tenuto fede alla parola data. in ..."
3,"Madam President, I really am quite astonished at Mr Barón Crespo' s behaviour and the fact that he is now asking for...","madam president, i really am quite astonished at mr barón crespo' s behaviour and the fact that he is now asking for...","Signora Presidente, onorevoli colleghi, sono piuttosto sorpreso del comportamento del collega, onorevole Barón Cresp...","signora presidente, onorevoli colleghi, sono piuttosto sorpreso del comportamento del collega, onorevole barón cresp..."
4,"I was told that large sections of the Socialist Group were also keen to have this item taken off the agenda, because...","i was told that large sections of the socialist group were also keen to have this item taken off the agenda, because...",Mi è stato detto che anche una parte cospicua del gruppo socialista vorrebbe che questo punto venisse ritirato dall'...,mi è stato detto che anche una parte cospicua del gruppo socialista vorrebbe che questo punto venisse ritirato dall'...



## 6. Check length changes

Lowercasing does not change token counts, but whitespace normalization and pair removal can slightly change the usable dataset size. These statistics are saved for the report and for Task 3 decisions such as maximum sequence length.


In [6]:
def token_lengths(texts: pd.Series) -> np.ndarray:
    """Compute whitespace-token lengths for a text series."""
    return texts.map(lambda s: len(s.split())).to_numpy()


length_stats = pd.DataFrame({
    "language": ["English", "Italian"],
    "mean_tokens": [
        token_lengths(clean_df["en"]).mean(),
        token_lengths(clean_df["it"]).mean(),
    ],
    "median_tokens": [
        np.median(token_lengths(clean_df["en"])),
        np.median(token_lengths(clean_df["it"])),
    ],
    "p90_tokens": [
        np.percentile(token_lengths(clean_df["en"]), 90),
        np.percentile(token_lengths(clean_df["it"]), 90),
    ],
    "p95_tokens": [
        np.percentile(token_lengths(clean_df["en"]), 95),
        np.percentile(token_lengths(clean_df["it"]), 95),
    ],
    "max_tokens": [
        token_lengths(clean_df["en"]).max(),
        token_lengths(clean_df["it"]).max(),
    ],
})

numeric_cols = length_stats.select_dtypes(include=["float", "int"]).columns
length_stats[numeric_cols] = length_stats[numeric_cols].round(2)

length_stats

,language,mean_tokens,median_tokens,p90_tokens,p95_tokens,max_tokens
0,English,26.12,24.0,46.0,54.0,320
1,Italian,25.21,23.0,44.0,53.0,298



## 7. Save processed data

The next tasks should use these outputs:

- `data/processed/task2_clean.en`
- `data/processed/task2_clean.it`

The CSV and JSON files are only for inspection and reporting.


In [7]:
EN_OUT = PROCESSED_DIR / "task2_clean.en"
IT_OUT = PROCESSED_DIR / "task2_clean.it"
CSV_OUT = PROCESSED_DIR / "task2_clean_parallel.csv"
SUMMARY_OUT = RESULTS_DIR / "task2_preprocessing_summary.csv"
LENGTH_OUT = RESULTS_DIR / "task2_length_stats.csv"
CONFIG_OUT = RESULTS_DIR / "task2_preprocessing_config.json"

def write_lines(path: Path, lines) -> None:
    """Write one sentence per line without CSV quoting or escaping."""
    with path.open("w", encoding="utf-8", newline="\n") as file:
        for line in lines:
            file.write(line + "\n")


write_lines(EN_OUT, clean_df["en"])
write_lines(IT_OUT, clean_df["it"])

clean_df[["en", "it"]].to_csv(CSV_OUT, index=False, encoding="utf-8")
summary.to_csv(SUMMARY_OUT, encoding="utf-8")
length_stats.to_csv(LENGTH_OUT, index=False, encoding="utf-8")

config = {
    "input_english": str(en_input_path.relative_to(ROOT)),
    "input_italian": str(it_input_path.relative_to(ROOT)),
    "output_english": str(EN_OUT.relative_to(ROOT)),
    "output_italian": str(IT_OUT.relative_to(ROOT)),
    "lowercase": LOWERCASE,
    "remove_empty_pairs": REMOVE_EMPTY_PAIRS,
    "remove_xml_tag_pairs": REMOVE_XML_TAG_PAIRS,
    "normalize_whitespace": True,
    "remove_punctuation": False,
    "remove_numbers": False,
    "remove_stopwords": False,
    "stemming_or_lemmatization": False,
    "pairs_before": int(len(df)),
    "pairs_after": int(len(clean_df)),
    "pairs_removed": int(len(df) - len(clean_df)),
}

with open(CONFIG_OUT, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Saved files:")
for path in [EN_OUT, IT_OUT, CSV_OUT, SUMMARY_OUT, LENGTH_OUT, CONFIG_OUT]:
    print(f"- {path.relative_to(ROOT)}")

Saved files:
- data/processed/task2_clean.en
- data/processed/task2_clean.it
- data/processed/task2_clean_parallel.csv
- results/task2_outputs/task2_preprocessing_summary.csv
- results/task2_outputs/task2_length_stats.csv
- results/task2_outputs/task2_preprocessing_config.json



## 8. Quick sanity check

This cell verifies that the saved files can be read back and still contain aligned sentence pairs.


In [8]:
en_check = read_lines(EN_OUT)
it_check = read_lines(IT_OUT)

# Alignment: same number of lines on both sides.
assert len(en_check) == len(it_check)
assert len(en_check) == len(clean_df)

# Content round-trip: what we saved must read back EXACTLY as the cleaned text.
# A length-only check cannot catch corruption of the line CONTENT - e.g. the
# earlier bug where saving via to_csv wrapped comma-containing lines in quotes.
# Comparing the reloaded lines against clean_df catches that whole class of bug.
# (Note: a handful of lines are genuinely quoted sentences in Europarl, so we do
# NOT test for leading/trailing quotes - only that the round-trip is exact.)
assert en_check == clean_df["en"].tolist(), "English round-trip mismatch (content changed on save)"
assert it_check == clean_df["it"].tolist(), "Italian round-trip mismatch (content changed on save)"
assert all(s.strip() for s in en_check) and all(s.strip() for s in it_check), "blank line saved"

print(f"Saved aligned pairs: {len(en_check):,}  (content round-trip verified)")
print("\nExample processed pairs:")

for i in range(min(3, len(en_check))):
    print(f"\n[{i}]")
    print("EN:", en_check[i])
    print("IT:", it_check[i])

Saved aligned pairs: 190,110  (content round-trip verified)

Example processed pairs:

[0]
EN: madam president, on a point of order.
IT: signora presidente, un richiamo al regolamento.

[1]
EN: madam president, coinciding with this year' s first part-session of the european parliament, a date has been set, unfortunately for next thursday, in texas in america, for the execution of a young 34 year-old man who has been sentenced to death. we shall call him mr hicks.
IT: signora presidente, in coincidenza con la prima tornata dell'anno del parlamento europeo, negli stati uniti in texas è stata fissata, purtroppo per giovedì prossimo, l'esecuzione di un condannato a morte, un giovane di 34 anni che chiameremo di nome hicks.

[2]
EN: madam president, i would firstly like to compliment you on the fact that you have kept your word and that, during this first part-session of the new year, the number of television channels in our offices has indeed increased considerably.
IT: signora presidente,


## 9. Critical discussion for the report

The preprocessing strategy is intentionally conservative because the downstream task is machine translation. We removed only clear noise at the aligned-pair level: empty lines and XML-like lines starting with `<`, because they do not provide useful parallel translation examples and would break the source--target correspondence. We lowercased the text to reduce unnecessary vocabulary variation between forms such as `The` and `the`, and we normalized repeated whitespace to make tokenization more stable.

We did not remove punctuation because punctuation contributes to sentence boundaries, pauses, and target-side fluency. We did not remove numbers because numerical expressions are often meaningful in parliamentary proceedings and should be translated or copied correctly. We also kept stop words, since function words are essential in translation and removing them would damage grammatical structure. Finally, we did not apply stemming or lemmatization because the model must learn to generate correctly inflected surface forms, especially in Italian, rather than reduced lemmas or stems.

Overall, this preprocessing cleans obvious noise while preserving the linguistic information needed by the sequence-to-sequence translation models in the next tasks.
